In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from keras.src.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.src.models import Sequential
from keras.src.callbacks import EarlyStopping
from keras.src.applications.resnet import ResNet50
from keras.src.applications.mobilenet_v3 import MobileNetV3Small
import sys
sys.path.append("../Handlers")
import preprocessing
from functools import partial
import url_preprocessing
from PIL import Image
import os

In [ ]:
class MultimodalClassifier:
    def __init__(self):
        self.text_model = None
        self.image_model = None
        self.url_model = None
        self.image_dataset = None

    def train(self, text_data, text_labels, image_data, image_labels, url_data, url_labels):
        print("Begin training text")
        self.build_text_model()

        X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
            text_data, text_labels, test_size=0.2, random_state=42
        )

        self.text_model.fit(X_text_train, y_text_train)
        text_pred = self.text_model.predict(X_text_test)
        text_accuracy = accuracy_score(y_text_test, text_pred)
        print(f"Text model accuracy: {text_accuracy:.4f}")

        print("Begin training image")
        self.build_image_model()

        X_image_train, X_image_test, y_image_train, y_image_test = train_test_split(
            image_data, image_labels, test_size=0.2, random_state=42
        )

        history = self.image_model.fit(
            X_image_train, y_image_train,
            validation_data=(X_image_test, y_image_test),
            epochs=10, batch_size=32, verbose=1,
            callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
        )

        img_pred = self.image_model.predict(X_image_test)
        img_pred_classes = (img_pred > 0.5).astype(int)
        img_accuracy = accuracy_score(y_image_test, img_pred_classes)
        print(f"Image model accuracy: {img_accuracy:.4f}")

        print("Begin training URL")
        self.build_url_model()

        X_url_train, X_url_test, y_url_train, y_url_test = train_test_split(
            url_data, url_labels, test_size=0.2, random_state=42
        )

        self.url_model.fit(X_url_train, y_url_train)
        url_pred = self.url_model.predict(X_url_test)
        url_accuracy = accuracy_score(y_url_test, url_pred)
        print(f"URL model accuracy: {url_accuracy:.4f}")

    def predict(self, text_data=None, image_data=None, url_data=None):
        predictions = []

        if text_data is not None and self.text_model is not None:
            text_pred = self.text_model.predict(text_data)
            predictions.append(text_pred)

        if image_data is not None and self.image_model is not None:
            img_pred = self.image_model.predict(image_data)
            img_pred_classes = (img_pred > 0.5).astype(int)
            predictions.append(img_pred_classes)

        if url_data is not None and self.url_model is not None:
            url_pred = self.url_model.predict(url_data)
            predictions.append(url_pred)

        

        stacked_preds = np.stack(predictions, axis=0)

        return (np.mean(stacked_preds, axis=0) > 0.5).astype(int)
    
    def preprocess_text_data(self, text_data):
        preprocession = partial(
            preprocessing.preprocess_text,
            remove_numbers=True
        )
        if isinstance(text_data, pd.DataFrame):
            text_data["preprocessed"] = text_data.apply(preprocession)

        return text_data
    
    def preprocess_url_data(self, url_data:pd.DataFrame):
        url_data, _ = url_preprocessing.cleanup_url_dataset(url_data)

        url_data = url_preprocessing.extract_url_features(url_data)

        return url_data
    
    def preprocess_image_data(self, image_0_path, image_1_path, save=True, X_save_path="./data/data_X.npy",
            y_save_path="./data/data_y.npy",
            img_size=(224, 224)):
        
        print("Loading and preprocessing data...")
        
        data = []
        
        # Load spam images (label 1)
        for filename in os.listdir(image_1_path):
            path = os.path.join(image_1_path, filename)
            try:
                img = Image.open(path).convert('RGB').resize(img_size)
                data.append((np.array(img), 1))
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        
        # Load natural images (label 0)
        for filename in os.listdir(image_0_path):
            path = os.path.join(image_0_path, filename)
            try:
                img = Image.open(path).convert('RGB').resize(img_size)
                data.append((np.array(img), 0))
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        
        X, y = zip(*data)

        if save:
            print("Saving np data")
            np.save(X_save_path, X)
            np.save(y_save_path, y)
        
        print(f"Loaded {len(X)} images")
        
        return X, y

    def build_text_model(self):
        self.text_model = VotingClassifier([
            ("dt", DecisionTreeClassifier()),
            ('rf', RandomForestClassifier(random_state=42, n_jobs=4)),
            ('ext', ExtraTreesClassifier(n_jobs=4, random_state=42)),
            ('svc', BaggingClassifier(n_jobs=4, random_state=42))
        ], voting="soft", n_jobs=2)

    def build_url_model(self):
        self.url_model = VotingClassifier([
            ("dt", DecisionTreeClassifier()),
            ('rf', RandomForestClassifier(random_state=42, n_jobs=4)),
            ('ext', ExtraTreesClassifier(n_jobs=4, random_state=42)),
            ('svc', BaggingClassifier(n_jobs=4, random_state=42))
        ], voting="soft", n_jobs=2)

    def build_image_model(self):
        base_model = MobileNetV3Small(
            include_top=False,
            input_shape=(224, 224, 3)
        )
        
        self.image_model = Sequential([
            base_model,
            GlobalAveragePooling2D(),
            Dropout(0.5),
            Dense(128, activation='relu'),
            Dropout(0.3),
            Dense(1, activation='sigmoid')
        ])

        self.image_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [3]:
import joblib

text_lemmatized_tfidf = joblib.load('./text_data/enron_lemmatized_tfidf.pkl')
text_X = text_lemmatized_tfidf['features']
text_y = text_lemmatized_tfidf['labels']

In [4]:
image_X = np.load('./image_data/data_X.npy')
image_y = np.load('./image_data/data_y.npy')

In [5]:
import pandas as pd
df = pd.read_csv("./url_data/malicious_phish_preprocessed_100k.csv")
url_X = df.drop(columns=["Unnamed: 0", "Unnamed: 0.1", "url", "type"])
url_y = df["type"]
del df

In [6]:
url_X = pd.get_dummies(url_X, columns=["tld"], drop_first=True)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
url_y = le.fit_transform(url_y)

url_y = (url_y > 0).astype(int)

In [7]:
mmc = MultimodalClassifier()

In [8]:
mmc.train(text_X, text_y, image_X, image_y, url_X, url_y)

Begin training text
Text model accuracy: 0.9745
Begin training image
Epoch 1/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 48s 587ms/step - accuracy: 0.9072 - loss: 0.2190 - val_accuracy: 0.8822 - val_loss: 0.7191
Epoch 2/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 557ms/step - accuracy: 0.9941 - loss: 0.0112 - val_accuracy: 0.8736 - val_loss: 1.0635
Epoch 3/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 544ms/step - accuracy: 0.9929 - loss: 0.0191 - val_accuracy: 0.8908 - val_loss: 1.2770
Epoch 4/10
44/44 ━━━━━━━━━━━━━━━━━━━━ 24s 549ms/step - accuracy: 0.9986 - loss: 0.0077 - val_accuracy: 0.7672 - val_loss: 3.0145
11/11 ━━━━━━━━━━━━━━━━━━━━ 4s 189ms/step
Image model accuracy: 0.8822
Begin training URL
URL model accuracy: 0.8910


In [11]:
text_X_test, _, text_y_test, _ = train_test_split(text_X, text_y, random_state=42, train_size=2000)

In [12]:
url_X_test, _, url_y_test, _ = train_test_split(url_X, url_y, random_state=42, train_size=2000)

url_y_test

In [13]:
preds = mmc.predict(text_data=text_X_test, url_data=url_X_test)

In [22]:
from sklearn.metrics import classification_report

print(len(text_y_test))
print(len(url_y_test))

print(classification_report(url_y_test, preds))

2000
2000
              precision    recall  f1-score   support

           0       0.44      0.81      0.57       648
           1       0.85      0.51      0.64      1352

    accuracy                           0.61      2000
   macro avg       0.65      0.66      0.60      2000
weighted avg       0.72      0.61      0.62      2000

